# Customer Churn Dataset — Data Quality Audit & Initial Patterns

**Objective:** Profile, clean, validate, summarize, and inspect a customer churn dataset.

This notebook documents the quality decisions made during the audit and highlights initial patterns. The dataset contains 15 customer records and 11 columns.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("../data/customer_churn_raw.csv")
df.head()


  CustomerID  Gender  Age  TenureMonths SubscriptionType  MonthlyCharges  TotalCharges    ContractType  SupportTickets  PaymentMethod Churn
0  CUST-1001  Female   34            12            Basic           49.99        599.88  Month-to-Month               3    Credit Card   Yes
1  CUST-1002    Male   45            24              Pro           79.99       1919.76        One Year               1  Bank Transfer    No
2  CUST-1003  Female   29             6            Basic           49.99        299.94  Month-to-Month               5            UPI   Yes
3  CUST-1004    Male   52            36       Enterprise          149.99       5399.64        Two Year               0    Credit Card    No
4  CUST-1005  Female   41             8              Pro           79.99        639.92  Month-to-Month               4     Debit Card   Yes

## 1. Dataset profile

We check:
- row and column counts
- column names
- data types
- missing values
- duplicate rows


In [2]:
print("Shape:", df.shape)
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())


Shape: (15, 11)

Data types:
CustomerID           object
Gender               object
Age                   int64
TenureMonths          int64
SubscriptionType     object
MonthlyCharges      float64
TotalCharges        float64
ContractType         object
SupportTickets        int64
PaymentMethod        object
Churn                object

Missing values:
CustomerID          0
Gender              0
Age                 0
TenureMonths        0
SubscriptionType    0
MonthlyCharges      0
TotalCharges        0
ContractType        0
SupportTickets      0
PaymentMethod       0
Churn               0

Duplicate rows: 0


## 2. Summary statistics

Numerical columns are summarized with descriptive statistics. Categorical columns are summarized using counts.


In [3]:
print(df.describe().T)
print("\nCategorical distributions:")
for col in df.select_dtypes(include="object").columns:
    print(f"\n{col}:")
    print(df[col].value_counts())


                count         mean          std    min     25%      50%      75%      max
Age              15.0    39.066667    11.392144  23.00   30.00    38.00    47.00    60.00
TenureMonths     15.0    18.800000    14.333776   2.00    7.00    15.00    27.00    48.00
MonthlyCharges   15.0    84.656667    42.739521  49.99   49.99    79.99   114.99   149.99
TotalCharges     15.0  2088.478667  2407.365767  99.98  449.91  1099.78  3209.73  7199.52
SupportTickets   15.0     2.533333     1.922300   0.00    1.00     2.00     4.00     6.00

Categorical distributions:
CustomerID:
CustomerID
CUST-1001    1
CUST-1002    1
CUST-1003    1
CUST-1004    1
CUST-1005    1
CUST-1006    1
CUST-1007    1
CUST-1008    1
CUST-1009    1
CUST-1010    1
CUST-1011    1
CUST-1012    1
CUST-1013    1
CUST-1014    1
CUST-1015    1
Gender:
Gender
Female    8
Male      7
SubscriptionType:
SubscriptionType
Basic         7
Pro           4
Enterprise    4
ContractType:
ContractType
Month-to-Month    7
One Year       

## 3. Outlier audit

For each numerical column, the IQR rule is used:

- Lower bound = Q1 − 1.5 × IQR
- Upper bound = Q3 + 1.5 × IQR

**Decision:** No numerical values fall outside these bounds, so no outliers are removed.


In [4]:
numeric_cols = df.select_dtypes(include=np.number).columns

for col in numeric_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = ((df[col] < lower) | (df[col] > upper)).sum()
    print(f"{col}: {count} outliers | bounds = ({lower:.2f}, {upper:.2f})")


Age: 0 outliers
TenureMonths: 0 outliers
MonthlyCharges: 0 outliers
TotalCharges: 0 outliers
SupportTickets: 0 outliers

## 4. Consistency check for TotalCharges

`TotalCharges` appears to be calculated from `MonthlyCharges × TenureMonths`.

**Decision:** Keep the supplied `TotalCharges` values because all 15 records match the expected calculation to 2 decimal places.


In [5]:
expected_total = (df["MonthlyCharges"] * df["TenureMonths"]).round(2)
mismatch = df["TotalCharges"].round(2) != expected_total

print("TotalCharges mismatches:", mismatch.sum())
print(pd.DataFrame({
    "CustomerID": df["CustomerID"],
    "TotalCharges": df["TotalCharges"],
    "ExpectedTotalCharges": expected_total,
    "Mismatch": mismatch
}))


TotalCharges mismatches: 0


## 5. Cleaning decisions

The audit found no missing values, duplicate rows, IQR outliers, or `TotalCharges` inconsistencies.

Therefore, **no rows or values were arbitrarily changed**. The cleaned dataset is a validated copy of the original dataset.

This is intentional: cleaning should fix demonstrated quality problems rather than change valid observations just to make the dataset look cleaner.


In [6]:
clean_df = df.copy()

print("Missing values after cleaning:", clean_df.isna().sum().sum())
print("Duplicate rows after cleaning:", clean_df.duplicated().sum())
print("Rows retained:", len(clean_df))


Missing values after cleaning: 0
Duplicate rows after cleaning: 0
Rows retained: 15


## 6. Churn distribution


In [7]:
churn_counts = clean_df["Churn"].value_counts()
churn_pct = clean_df["Churn"].value_counts(normalize=True).mul(100).round(1)

print(churn_counts)
print("\nPercentage:")
print(churn_pct)


Churn
No     8
Yes    7

Percentage:
Churn
No     53.3
Yes    46.7

## 7. Meaningful distributions


In [8]:
fig, ax = plt.subplots()
clean_df["Age"].plot(kind="hist", bins=6, ax=ax)
ax.set_title("Customer Age Distribution")
ax.set_xlabel("Age")
plt.show()

fig, ax = plt.subplots()
clean_df["TenureMonths"].plot(kind="hist", bins=6, ax=ax)
ax.set_title("Customer Tenure Distribution")
ax.set_xlabel("Tenure (months)")
plt.show()

fig, ax = plt.subplots()
clean_df["MonthlyCharges"].plot(kind="hist", bins=6, ax=ax)
ax.set_title("Monthly Charges Distribution")
ax.set_xlabel("Monthly Charges")
plt.show()


## 8. Initial churn patterns

Because this is a very small sample (15 customers), these findings are **descriptive only** and should not be treated as statistically generalizable.


In [9]:
print("Overall churn rate:", round((clean_df["Churn"].eq("Yes").mean()*100), 1), "%")

for col in ["SubscriptionType", "ContractType", "PaymentMethod"]:
    print(f"\nChurn rate by {col}:")
    print((clean_df.groupby(col)["Churn"].apply(lambda x: (x == "Yes").mean()*100)
           .round(1).sort_values(ascending=False)))

print("\nNumeric averages by churn:")
print(clean_df.groupby("Churn")[["Age","TenureMonths","MonthlyCharges","SupportTickets"]].mean().round(2))


Overall churn rate: 46.7 %

SubscriptionType:
Basic 71.4
Pro 50.0
Enterprise 0.0

ContractType:
Month-to-Month 100.0
One Year 0.0
Two Year 0.0

PaymentMethod:
Debit Card 100.0
UPI 75.0
Credit Card 33.3
Bank Transfer 0.0

Numeric averages by churn:
       Age  TenureMonths  MonthlyCharges  SupportTickets
Churn
No     39.38         29.12          107.49             1.00
Yes    38.71          7.00           58.56             4.29


## 9. Key observations

- **46.7%** of customers in this sample churned.
- Churned customers had much lower average tenure (**7.0 months**) than retained customers (**29.1 months**).
- Churned customers had more support tickets on average (**4.29**) than retained customers (**1.00**).
- Month-to-month customers show a **100% churn rate** in this tiny sample, while one-year and two-year customers show 0%.
- Basic subscribers show higher churn (71.4%) than Pro (50.0%) and Enterprise (0%) in this sample.
- These patterns are useful for exploration, but the dataset is too small to establish causal or population-level conclusions.


## 10. Final quality decision

**Status: PASS — dataset is already clean enough for exploratory analysis.**

No records were removed because there was no evidence of missingness, duplication, invalid numeric values, or inconsistent total charges. The cleaned CSV is saved separately for reproducibility.
